In [ ]:
!pip install praw pandas matplotlib 

In [ ]:
import pandas as pd
import sys
import json
import string
from collections import Counter
import nltk
import re
import html
#for messy text in comments, helps removes non-lexicon data for cleaner analysis
from nltk.tokenize import TweetTokenizer
tokenizer = TweetTokenizer()

from nltk.stem import PorterStemmer
stemmer = PorterStemmer()

nltk.download('stopwords')
import matplotlib.pyplot as mpl



In [ ]:
#Function to pre-process reddit data

def processText(text, tokenizer, stemmer, stopwords):

    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    lTokens = tokenizer.tokenize(text)
    lTokens = [token.strip() for token in lTokens]
    lStemmedTokens = [stemmer.stem(tok) for tok in lTokens]
    
    return [tok for tok in lStemmedTokens if tok not in stopwords and not tok.isdigit()]

    

In [ ]:
#doing tokenization stopword removal and stemming to furher clean the data

fJsonName = "rawComments.json"
dfComments = pd.read_json("rawComments.json")

#removing deleted and duplicated comments
dfComments = dfComments.dropna(subset=["body"])
dfComments = dfComments[~dfComments["body"].isin(["[deleted]", "[removed]"])]

wordFrequency = 50

lPunct = list(string.punctuation)
lStopwords = nltk.corpus.stopwords.words('english') + lPunct + ['via', 'http', 'https']

wordFreqCounter = Counter()

cleanComments = []

for comment in dfComments["body"]:

    lTokens = processText(
        text=str(comment),
        stopwords=lStopwords,
        tokenizer=tokenizer,
        stemmer=stemmer,
    )

    wordFreqCounter.update(lTokens)

    cleanComment = " ".join(lTokens)
    cleanComments.append(cleanComment)

#adding cleaned column
dfComments["cleanBody"] = cleanComments

#saving cleaned data
dfComments.to_json("cleanComments.json", orient="records")

#printing the most common words to check pre-processing worked
for word, count in wordFreqCounter.most_common(wordFrequency):
    print(word + ': ' + str(count)) 


In [ ]:
#Data graph for more frequent terms 
words = []
counts = []

for word, count in wordFreqCounter.most_common(wordFrequency):
    words.append(word)
    counts.append(count)

mpl.figure(figsize=(12, 6))
mpl.bar(words, counts)
mpl.xticks(rotation=90)
mpl.title("The Top 50 Most Frequent Words in Political Reddit comments")
mpl.xlabel("Words")
mpl.ylabel("Frequency")
mpl.tight_layout()
mpl.show()
